# Apples

### Import

In [ ]:
import os, sys, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller
from arch import arch_model 

warnings.filterwarnings("ignore")
sys.path.append(os.path.abspath('../scripts'))

from spread import SPREAD
from engine import ENGINE
from backtester import BACKTESTER
from tearsheet import TEARSHEET

### Data 

In [ ]:
months = [
    "202501", "202502", "202503", "202504", "202505", "202506",
    "202507", "202508", "202509", "202510", "202511", "202512"
]

my_files = [
    [f"../data/processed/audusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/audusd_dukascopy_bid_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_ask_{m}.parquet" for m in months],
    [f"../data/processed/nzdusd_dukascopy_bid_{m}.parquet" for m in months]
]

### SPREAD

In [ ]:
builder = SPREAD(threshold=1000) 
df = builder.build(my_files)

print(df.head(5))

### ENGINE

In [ ]:
print("⚙️ INITIATING WALK-FORWARD OOS PIPELINE...")

# 1. Prepare Date grouping
df['Date'] = df.index.date
unique_days = df['Date'].unique()

train_days = 30 
out_of_sample_results = []

print(f"Rolling Train Window: {train_days} days. Total Days: {len(unique_days)}")

# 2. The Walk-Forward Loop
for i in range(train_days, len(unique_days)):
    
    # Slice the chronological training and testing sets
    train_dates = unique_days[i - train_days : i]
    test_date = unique_days[i]
    
    train_df = df[df['Date'].isin(train_dates)].copy()
    test_df = df[df['Date'] == test_date].copy()
    
    # --- STEP A: FIT ON THE PAST ---
    engine = ENGINE(train_df)
    train_fitted = engine.fit_cointegration(z_window=1000)
    
    # We fit all three models here
    engine.fit_ar_reversion(lags=1)
    engine.fit_garch_vol(scaling=10000)
    engine.fit_markov_regimes(k_regimes=2)
    
    # --- STEP B: PREDICT THE FUTURE ---
    # Projects the frozen parameters onto the unseen test_df
    oos_predictions = engine.predict_oos(test_df, train_fitted, z_window=1000)
    out_of_sample_results.append(oos_predictions)
    
    if i % 10 == 0:
        print(f"✅ Processed up to {test_date} | Beta: {engine.beta:.4f} | GARCH Vol: {engine.forecasted_vol:.2f}")

# 3. Stitch results together
live_trading_data = pd.concat(out_of_sample_results)
print(f"\n🎉 OOS Dataset Built: {len(live_trading_data)} rows.")

### BACKTEST

In [ ]:
print("🚀 RUNNING BACKTEST...")

# 1. Run the strategy simulation
bt = BACKTESTER(live_trading_data)
# We use the HMM Danger Probability as our main kill-switch
results_df = bt.run(
    entry_z=2.0, 
    exit_z=0.0, 
    danger_threshold=0.5, 
    fee_bps=0.5
)

### TEARSHEEET

In [ ]:
print("\n--- GENERATING FINANCIAL TEARSHEET ---")
ts = TEARSHEET(results_df)
ts.calculate_financials()

# 3. Plot the Equity Curve and Drawdowns
ts.plot_performance()